In [2]:
"""
===============================================================================
END-TO-END INFERENCE PIPELINE FOR NEW-TO-CREDIT (NTC) BORROWER
===============================================================================
Input: new_bank_statement.csv (or dataset/new_bank_statement.csv)
Outputs:
  - outputs/credit_reports/new_borrower_credit_report.csv
  - outputs/credit_reports/new_borrower_credit_report.json
  - outputs/credit_reports/new_borrower_assessment_dashboard.png
===============================================================================
"""

import os
import re
import json
import joblib
import warnings
import logging
from pathlib import Path
from typing import Dict, Any, List, Tuple

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

warnings.filterwarnings('ignore')
sns.set_theme(style='whitegrid', palette='muted')
plt.rcParams['figure.figsize'] = (10, 6)
plt.rcParams['font.size'] = 11

LOGGER = logging.getLogger('ntc_borrower_scoring')
logging.basicConfig(level=logging.INFO, format='%(asctime)s | %(levelname)s | %(message)s')

OUTPUT_DIR = Path("./outputs/credit_reports")
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

# -----------------------------------------------------------------------------
# 1. LOAD PRE-TRAINED MODEL ARTIFACTS
# -----------------------------------------------------------------------------
fraud_path = Path("models/fraud_detection_pipeline.joblib")
credit_path = Path("models/credit_risk_pipeline.joblib")

if not fraud_path.exists() or not credit_path.exists():
    raise FileNotFoundError("Missing joblib model artifacts in 'models/' directory. Run model training first.")

fraud_payload = joblib.load(fraud_path)
credit_payload = joblib.load(credit_path)

fraud_model = fraud_payload['model'] if isinstance(fraud_payload, dict) and 'model' in fraud_payload else fraud_payload
credit_model = credit_payload['model'] if isinstance(credit_payload, dict) and 'model' in credit_payload else credit_payload

fraud_thresh = fraud_payload.get('threshold', 0.41) if isinstance(fraud_payload, dict) else 0.41
credit_thresh = credit_payload.get('threshold', 0.41) if isinstance(credit_payload, dict) else 0.41

print("✓ Pre-trained Fraud Detection & Credit Risk models loaded successfully.")

# -----------------------------------------------------------------------------
# HELPER: EXTRACT EXPECTED FEATURE NAMES FROM PIPELINE
# -----------------------------------------------------------------------------
def get_expected_features(model_pipeline, payload) -> List[str]:
    """Retrieves expected feature names from model payload or preprocessor."""
    if isinstance(payload, dict) and 'feature_columns' in payload and payload['feature_columns']:
        return list(payload['feature_columns'])
    if hasattr(model_pipeline, 'feature_names_in_'):
        return list(model_pipeline.feature_names_in_)
    if hasattr(model_pipeline, 'named_steps') and 'preprocess' in model_pipeline.named_steps:
        pre = model_pipeline.named_steps['preprocess']
        if hasattr(pre, 'feature_names_in_'):
            return list(pre.feature_names_in_)
    return []

fraud_expected_features = get_expected_features(fraud_model, fraud_payload)
credit_expected_features = get_expected_features(credit_model, credit_payload)

# -----------------------------------------------------------------------------
# 2. INGEST NEW BANK STATEMENT & PARSE NARRATION CATEGORIES
# -----------------------------------------------------------------------------
statement_path = Path("new_bank_statement.csv")
if not statement_path.exists():
    statement_path = Path("dataset/new_bank_statement.csv")

if not statement_path.exists():
    raise FileNotFoundError("Input file 'new_bank_statement.csv' or 'dataset/new_bank_statement.csv' not found.")

print(f"Reading raw bank statement CSV from '{statement_path}'...")
df_stmt = pd.read_csv(statement_path)

def categorize_narration(narr: str) -> str:
    if not isinstance(narr, str): return 'OTHERS'
    narr_upper = narr.upper()
    if re.search(r'SALARY|NETSALARY|PAYROLL', narr_upper): return 'SALARY'
    if re.search(r'ZERODHA|GROWW|MFAUTOPAY|PAYTMMONEY|MUTUAL|SIP', narr_upper): return 'INVESTMENTS_SIP'
    if re.search(r'BESCOM|TATAPOWER|AIRTEL|RECHARGE|UTILITY|BILL', narr_upper): return 'UTILITIES_BILLS'
    if re.search(r'BLINKIT|ZEPTO|BIGBASKET|GROCERY|SUPERMARKET', narr_upper): return 'GROCERY'
    if re.search(r'SWIGGY|ZOMATO|RESTAURANT|FOOD|CAFE', narr_upper): return 'FOOD_DELIVERY'
    if re.search(r'AMAZON|FLIPKART|MYNTRA|SHOPPING', narr_upper): return 'ECOMMERCE_SHOPPING'
    if re.search(r'UBER|OLA|IRCTC|FASTAG|TRAVEL', narr_upper): return 'TRAVEL_MOBILITY'
    if re.search(r'ATM-WDL|CASH|NFS', narr_upper): return 'ATM_CASH'
    if re.search(r'EMI|LOAN|BAJAJFINSERV|HOMEEMI|PERSONALEMI', narr_upper): return 'SHADOW_EMI'
    return 'OTHERS'

# -----------------------------------------------------------------------------
# 3. FEATURE EXTRACTION & AGGREGATION ENGINE
# -----------------------------------------------------------------------------
tx = df_stmt.copy()
for col in ['Withdrawal Amount', 'Deposit Amount', 'Closing Balance']:
    if col in tx.columns:
        tx[col] = pd.to_numeric(tx[col], errors='coerce').fillna(0.0)

tx['Date'] = pd.to_datetime(tx['Date'], errors='coerce')
tx['year_month'] = tx['Date'].dt.to_period('M').astype(str)
tx['is_debit'] = tx['Withdrawal Amount'].gt(0)
tx['is_credit'] = tx['Deposit Amount'].gt(0)
tx['debit_amt'] = np.where(tx['is_debit'], tx['Withdrawal Amount'], 0.0)
tx['credit_amt'] = np.where(tx['is_credit'], tx['Deposit Amount'], 0.0)
tx['amount'] = np.where(tx['is_credit'], tx['Deposit Amount'], tx['Withdrawal Amount'])
tx['category_parsed'] = tx['Narration'].apply(categorize_narration)

# Monthly aggregations
monthly = tx.groupby(['Customer_ID', 'year_month'], as_index=False).agg(
    monthly_income=('credit_amt', 'sum'),
    monthly_expense=('debit_amt', 'sum')
)
monthly['monthly_savings'] = monthly.monthly_income - monthly.monthly_expense

base = tx.groupby('Customer_ID').agg(
    transaction_frequency=('amount', 'size'),
    average_transaction_amount=('amount', 'mean'),
    median_transaction_amount=('amount', 'median'),
    largest_deposit=('Deposit Amount', 'max'),
    largest_withdrawal=('Withdrawal Amount', 'max'),
    average_balance=('Closing Balance', 'mean'),
    minimum_balance=('Closing Balance', 'min'),
    maximum_balance=('Closing Balance', 'max'),
    balance_variance=('Closing Balance', 'var'),
    transaction_consistency=('amount', lambda x: 1 / (1 + x.std(ddof=0) / (x.mean() + 1)))
)

monthly_f = monthly.groupby('Customer_ID').agg(
    average_monthly_income=('monthly_income', 'mean'),
    income_consistency=('monthly_income', lambda x: 1 / (1 + x.std(ddof=0) / (x.mean() + 1))),
    average_monthly_expense=('monthly_expense', 'mean'),
    monthly_savings=('monthly_savings', 'mean'),
    expense_stability=('monthly_expense', lambda x: 1 / (1 + x.std(ddof=0) / (x.mean() + 1)))
)

sums = tx.groupby('Customer_ID').agg(
    total_income=('credit_amt', 'sum'),
    total_expense=('debit_amt', 'sum'),
    essential_expense=('debit_amt', lambda x: x[tx.loc[x.index, 'category_parsed'].isin(['UTILITIES_BILLS', 'GROCERY'])].sum()),
    discretionary_expense=('debit_amt', lambda x: x[tx.loc[x.index, 'category_parsed'].isin(['FOOD_DELIVERY', 'ECOMMERCE_SHOPPING', 'TRAVEL_MOBILITY'])].sum()),
    atm_withdrawals=('debit_amt', lambda x: x[tx.loc[x.index, 'category_parsed'] == 'ATM_CASH'].sum()),
    emi_payments=('debit_amt', lambda x: x[tx.loc[x.index, 'category_parsed'] == 'SHADOW_EMI'].sum()),
    investment_amount=('debit_amt', lambda x: x[tx.loc[x.index, 'category_parsed'] == 'INVESTMENTS_SIP'].sum()),
    salary_credits=('credit_amt', lambda x: x[tx.loc[x.index, 'category_parsed'] == 'SALARY'].sum())
)

df_extracted = base.join([monthly_f, sums], how='left').fillna(0).reset_index()

def safe_divide(n, d):
    num = np.asarray(n, dtype=float)
    den = np.asarray(d, dtype=float)
    out = np.zeros(np.broadcast_shapes(num.shape, den.shape), dtype=float)
    return np.divide(num, den, out=out, where=den != 0)

df_extracted['expense_ratio'] = safe_divide(df_extracted.total_expense, df_extracted.total_income)
df_extracted['savings_ratio'] = safe_divide(df_extracted.monthly_savings, df_extracted.average_monthly_income)
df_extracted['essential_expense_ratio'] = safe_divide(df_extracted.essential_expense, df_extracted.total_expense)
df_extracted['discretionary_expense_ratio'] = safe_divide(df_extracted.discretionary_expense, df_extracted.total_expense)
df_extracted['atm_cash_ratio'] = safe_divide(df_extracted.atm_withdrawals, df_extracted.total_expense)
df_extracted['investment_ratio'] = safe_divide(df_extracted.investment_amount, df_extracted.total_income)
df_extracted['salary_ratio'] = safe_divide(df_extracted.salary_credits, df_extracted.total_income)
df_extracted['financial_buffer'] = safe_divide(df_extracted.minimum_balance.clip(lower=0), df_extracted.average_monthly_expense)
df_extracted['income_stability_index'] = df_extracted['income_consistency'] * 0.6 + df_extracted['expense_stability'] * 0.4
df_extracted['digital_payment_ratio'] = 1.0 - df_extracted['atm_cash_ratio']

print(f"✓ Engineered {df_extracted.shape[1]} features from narration text for Customer ID: {df_extracted['Customer_ID'].iloc[0]}")

# -----------------------------------------------------------------------------
# 4. SAFE INFERENCE MATRIX ALIGNMENT & DUMMY INJECTION (PREVENTS VALUEERROR)
# -----------------------------------------------------------------------------
def align_and_fill_features(df_in: pd.DataFrame, expected_cols: List[str]) -> pd.DataFrame:
    df_aligned = df_in.copy()
    
    if not expected_cols:
        drop_cols = ['Customer_ID', 'customer_id']
        df_aligned = df_aligned.drop(columns=[c for c in drop_cols if c in df_aligned.columns])
        for c in df_aligned.select_dtypes(include=['object', 'category']).columns:
            df_aligned[c] = df_aligned[c].astype('category').cat.codes + 1
        return df_aligned.fillna(0.0)

    # Ingest default values for any missing columns expected by model preprocessor
    for col in expected_cols:
        if col not in df_aligned.columns:
            col_lower = col.lower()
            if 'credit_score' in col_lower:
                df_aligned[col] = 650.0
            elif 'match_score' in col_lower or 'liveness' in col_lower:
                df_aligned[col] = 0.90
            elif 'kyc' in col_lower or 'verified' in col_lower:
                df_aligned[col] = 1.0
            else:
                df_aligned[col] = 0.0

    X_out = df_aligned[expected_cols].copy()
    
    for c in X_out.select_dtypes(include=['object', 'category']).columns:
        X_out[c] = X_out[c].astype('category').cat.codes + 1

    return X_out.fillna(0.0)

X_inf_fraud = align_and_fill_features(df_extracted, fraud_expected_features)
X_inf_credit = align_and_fill_features(df_extracted, credit_expected_features)

# -----------------------------------------------------------------------------
# 5. SUBSYSTEM INFERENCE (FRAUD & DEFAULT RISK)
# -----------------------------------------------------------------------------
fraud_prob = float(fraud_model.predict_proba(X_inf_fraud)[:, 1][0])
pd_prob = float(credit_model.predict_proba(X_inf_credit)[:, 1][0])

def assign_fraud_risk(prob: float) -> str:
    if prob >= 0.60: return "CRITICAL"
    if prob >= 0.25: return "SUSPICIOUS"
    return "CLEAN"

GRADE_BOUNDS = {'MAX_A': 0.15, 'MAX_B': 0.30, 'MAX_C': 0.50, 'MAX_D': 0.70}

def map_pd_to_regulatory_grade(pd_val: float, bounds: Dict[str, float]) -> str:
    if pd_val <= bounds['MAX_A']: return 'Grade A'
    if pd_val <= bounds['MAX_B']: return 'Grade B'
    if pd_val <= bounds['MAX_C']: return 'Grade C'
    if pd_val <= bounds['MAX_D']: return 'Grade D'
    return 'Grade E'

fraud_level = assign_fraud_risk(fraud_prob)
risk_grade = map_pd_to_regulatory_grade(pd_prob, GRADE_BOUNDS)

# -----------------------------------------------------------------------------
# 6. UNDERWRITING & COMPOSITE AI CREDIT SCORING ENGINE
# -----------------------------------------------------------------------------
row = df_extracted.iloc[0].to_dict()

monthly_income = float(row.get('average_monthly_income', 30000))
monthly_expense = float(row.get('average_monthly_expense', 15000))
existing_emi = float(row.get('emi_payments', 0)) / 12.0

available_disposable_income = monthly_income - monthly_expense - existing_emi
foir = (existing_emi / monthly_income) if monthly_income > 0 else 1.0

pricing_matrix = {
    'Grade A': {"base_rate": 0.105, "multiplier": 8.0},
    'Grade B': {"base_rate": 0.120, "multiplier": 6.0},
    'Grade C': {"base_rate": 0.145, "multiplier": 4.0},
    'Grade D': {"base_rate": 0.180, "multiplier": 2.0},
    'Grade E': {"base_rate": 0.240, "multiplier": 1.0},
}

tier = pricing_matrix.get(risk_grade, pricing_matrix['Grade E'])
max_approved_loan = int(max(0, monthly_income * tier["multiplier"]))
recommended_loan = int(max_approved_loan * 0.80)

if fraud_prob >= 0.75:
    decision_str = "REJECTED"
    reason_str = "Severe fraud risk flag triggered."
    max_approved_loan = 0
    recommended_loan = 0
elif pd_prob >= 0.80:
    decision_str = "REJECTED"
    reason_str = "Very high probability of default."
    max_approved_loan = 0
    recommended_loan = 0
elif pd_prob >= 0.50 or fraud_prob >= 0.50 or risk_grade in {'Grade D', 'Grade E'}:
    decision_str = "REVIEW"
    reason_str = "Manual review recommended due to elevated risk parameters."
    recommended_loan = int(max_approved_loan * 0.50)
elif foir > 0.45 or available_disposable_income <= 0:
    decision_str = "REVIEW"
    reason_str = "Manual review recommended due to cashflow margin or EMI burden."
    recommended_loan = int(max_approved_loan * 0.50)
else:
    decision_str = "APPROVED"
    reason_str = "Passed alternative risk underwriting policy based on verified bank statement cashflow."

pd_factor = (1.0 - pd_prob) * 45
stability_factor = min(1.0, float(row['income_stability_index'])) * 30
fraud_factor = (1.0 - fraud_prob) * 15
savings_factor = max(0.0, min(1.0, float(row['savings_ratio']) + 0.5)) * 10

ai_credit_score = int(np.clip(pd_factor + stability_factor + fraud_factor + savings_factor, 0, 100))

recs = []
if float(row['savings_ratio']) < 0.20:
    recs.append("Increase your monthly savings cushion above 20% of net monthly income.")
if float(row['atm_cash_ratio']) > 0.20:
    recs.append("Reduce liquid cash withdrawals to improve digital financial traceability.")
if float(row['expense_ratio']) > 0.90:
    recs.append("Reduce discretionary e-commerce and dining spending to build a cash reserve.")

recommendations_str = " | ".join(recs) if recs else "Maintain current positive credit and balance management behaviors."

biz_explanation = f"Evaluated new borrower '{row['Customer_ID']}'. AI Credit Score: {ai_credit_score}/100. Fraud Probability: {fraud_prob:.2%}, Probability of Default: {pd_prob:.2%}. Underwriting Decision: {decision_str}."
customer_explanation = f"Your application was evaluated using your bank statement cashflow data. Your alternative AI credit score is {ai_credit_score}/100 with a {risk_grade} risk classification."

report_data = {
    "CustomerID": row['Customer_ID'],
    "AI_Credit_Score": ai_credit_score,
    "Income_Stability_Score": round(float(row['income_stability_index']), 4),
    "Fraud_Probability": round(fraud_prob, 4),
    "Probability_of_Default": round(pd_prob, 4),
    "Risk_Grade": risk_grade,
    "Underwriting_Decision": decision_str,
    "Decision_Reason": reason_str,
    "Max_Approved_Limit": max_approved_loan,
    "Recommended_Loan_Amount": recommended_loan,
    "Interest_Rate": tier["base_rate"],
    "Business_Explanation": biz_explanation,
    "Customer_Explanation": customer_explanation,
    "Actionable_Recommendations": recommendations_str,
    "income_consistency": round(float(row['income_consistency']), 4),
    "expense_ratio": round(float(row['expense_ratio']), 4),
    "savings_ratio": round(float(row['savings_ratio']), 4)
}

# -----------------------------------------------------------------------------
# 7. EXPORT REPORTS & GENERATE DASHBOARD VISUALIZATIONS
# -----------------------------------------------------------------------------
df_report = pd.DataFrame([report_data])
df_report.to_csv(OUTPUT_DIR / 'new_borrower_credit_report.csv', index=False)
with open(OUTPUT_DIR / 'new_borrower_credit_report.json', 'w') as f:
    json.dump(report_data, f, indent=4)

print(f"\n[✓] Report CSV saved to: {OUTPUT_DIR / 'new_borrower_credit_report.csv'}")
print(f"[✓] Report JSON saved to: {OUTPUT_DIR / 'new_borrower_credit_report.json'}")

fig, axes = plt.subplots(2, 2, figsize=(14, 10))

# Panel 1: Monthly Cashflow Dynamics
tx_monthly = tx.groupby('year_month').agg(
    Monthly_Income=('credit_amt', 'sum'),
    Monthly_Expense=('debit_amt', 'sum')
).reset_index()

axes[0, 0].plot(tx_monthly['year_month'], tx_monthly['Monthly_Income'], marker='o', color='#2ecc71', linewidth=2.5, label='Inflow (Salary & Credits)')
axes[0, 0].plot(tx_monthly['year_month'], tx_monthly['Monthly_Expense'], marker='s', color='#e74c3c', linewidth=2.5, label='Outflow (Expenses & Debits)')
axes[0, 0].set_title('Monthly Cashflow Dynamics (2025 - 2026)', fontweight='bold', fontsize=12)
axes[0, 0].set_xlabel('Month', fontsize=10)
axes[0, 0].set_ylabel('Amount (INR)', fontsize=10)
axes[0, 0].tick_params(axis='x', rotation=45)
axes[0, 0].legend()
axes[0, 0].grid(True, linestyle='--', alpha=0.5)

# Panel 2: Spending Breakdown
cat_totals = tx.groupby('category_parsed')['debit_amt'].sum().sort_values(ascending=False)
palette = sns.color_palette("viridis", len(cat_totals))
axes[0, 1].barh(cat_totals.index, cat_totals.values, color=palette)
axes[0, 1].set_title('Narration-Parsed Expense Distribution', fontweight='bold', fontsize=12)
axes[0, 1].set_xlabel('Total Spent (INR)', fontsize=10)
for index, value in enumerate(cat_totals.values):
    axes[0, 1].text(value, index, f" ₹{value:,.0f}", va='center', fontsize=9)

# Panel 3: Risk Metrics Gauge
metrics = ['AI Credit Score\n(0-100)', 'Income Stability\n(0-100)', 'Fraud Risk\n(%)', 'Default Risk (PD)\n(%)']
scores = [ai_credit_score, round(row['income_stability_index']*100, 1), round(fraud_prob*100, 1), round(pd_prob*100, 1)]
colors = ['#3498db', '#2ecc71', '#f39c12', '#e74c3c']

bars = axes[1, 0].bar(metrics, scores, color=colors, width=0.5)
axes[1, 0].set_title('Model Inferred Risk Indicators & Score', fontweight='bold', fontsize=12)
axes[1, 0].set_ylim(0, 110)
for bar in bars:
    yval = bar.get_height()
    axes[1, 0].text(bar.get_x() + bar.get_width()/2.0, yval + 2, f"{yval:.1f}", ha='center', va='bottom', fontweight='bold')

# Panel 4: Underwriting Decision Card
axes[1, 1].axis('off')
summary_text = (
    f"NEW BORROWER UNDERWRITING SUMMARY\n"
    f"-----------------------------------------\n"
    f"• Customer ID          : {row['Customer_ID']}\n"
    f"• Total Txns Parsed    : {len(df_stmt):,}\n"
    f"• Avg Monthly Income   : ₹{row['average_monthly_income']:,.2f}\n"
    f"• AI Credit Score      : {ai_credit_score} / 100\n"
    f"• Fraud Probability    : {fraud_prob:.2%} ({fraud_level})\n"
    f"• Default Prob. (PD)   : {pd_prob:.2%} ({risk_grade})\n"
    f"-----------------------------------------\n"
    f"• Underwriting Status  : {decision_str}\n"
    f"• Decision Reason      : {reason_str}\n"
    f"• Max Approved Limit   : ₹{max_approved_loan:,.0f}\n"
    f"• Rec. Loan Amount     : ₹{recommended_loan:,.0f}\n"
    f"• Applicable Base Rate : {tier['base_rate']*100:.1f}% p.a.\n"
)
axes[1, 1].text(0.05, 0.95, summary_text, transform=axes[1, 1].transAxes, fontsize=11,
                verticalalignment='top', bbox=dict(boxstyle='round,pad=0.8', facecolor='#f8f9fa', edgecolor='#bdc3c7'))

plt.tight_layout()
dashboard_path = OUTPUT_DIR / 'new_borrower_assessment_dashboard.png'
plt.savefig(dashboard_path, dpi=300, bbox_inches='tight')
plt.close()

print(f"[✓] Assessment dashboard image saved to: {dashboard_path}")
print("\n=== FINAL OUTPUT JSON ===")
print(json.dumps(report_data, indent=4))


2026-08-30 23:50:15,898 | INFO | Using categorical units to plot a list of strings that are all parsable as floats or dates. If these strings should be plotted as numbers, cast to the appropriate data type before plotting.
2026-08-30 23:50:15,898 | INFO | Using categorical units to plot a list of strings that are all parsable as floats or dates. If these strings should be plotted as numbers, cast to the appropriate data type before plotting.
2026-08-30 23:50:15,899 | INFO | Using categorical units to plot a list of strings that are all parsable as floats or dates. If these strings should be plotted as numbers, cast to the appropriate data type before plotting.
2026-08-30 23:50:15,900 | INFO | Using categorical units to plot a list of strings that are all parsable as floats or dates. If these strings should be plotted as numbers, cast to the appropriate data type before plotting.


✓ Pre-trained Fraud Detection & Credit Risk models loaded successfully.
Reading raw bank statement CSV from 'dataset/new_bank_statement.csv'...
✓ Engineered 34 features from narration text for Customer ID: CUST_00111111

[✓] Report CSV saved to: outputs/credit_reports/new_borrower_credit_report.csv
[✓] Report JSON saved to: outputs/credit_reports/new_borrower_credit_report.json
[✓] Assessment dashboard image saved to: outputs/credit_reports/new_borrower_assessment_dashboard.png

=== FINAL OUTPUT JSON ===
{
    "CustomerID": "CUST_00111111",
    "AI_Credit_Score": 80,
    "Income_Stability_Score": 0.8472,
    "Fraud_Probability": 0.013,
    "Probability_of_Default": 0.2107,
    "Risk_Grade": "Grade B",
    "Underwriting_Decision": "REVIEW",
    "Decision_Reason": "Manual review recommended due to cashflow margin or EMI burden.",
    "Max_Approved_Limit": 1118002,
    "Recommended_Loan_Amount": 559001,
    "Interest_Rate": 0.12,
    "Business_Explanation": "Evaluated new borrower 'CUST_0